# Label-Free Free-Energy Selective Prediction with Hybrid Quantum Latent Features

This corrected Step-2 notebook implements the experimental workflow for pneumonia triage from PneumoniaMNIST. It removes label leakage from the free-energy score, locks operational thresholds on the validation set, corrects calibration analysis, restores the best validation checkpoint, and saves per-case outputs for independent audit.

Three models are supported: a latent-only baseline, a capacity-aligned classical nonlinear control, and the simulated latent--quantum hybrid. The classical control is essential for testing whether any hybrid gain is attributable specifically to the parameterized quantum feature map rather than merely to feature augmentation.

> **Important:** Numerical values and manuscript figures from the earlier notebook must be regenerated. The previous `correct_nll` term used the true test label and therefore could not define a deployment-time accept--refer policy.


## 1. Environment setup and reproducibility

The cell below imports all dependencies, fixes random seeds, configures deterministic behavior where supported, and creates a project-specific output directory. In Google Colab, Google Drive is mounted automatically so that figures, tables, checkpoints, and logs persist after the session.


In [ ]:
# In a fresh Colab runtime, run this installation command once:
# !pip -q install medmnist pennylane scikit-learn

import copy
import json
import math
import os
import platform
import random
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset

from sklearn.metrics import accuracy_score, balanced_accuracy_score, brier_score_loss, f1_score, roc_auc_score, roc_curve
from sklearn.manifold import TSNE
from sklearn.model_selection import StratifiedShuffleSplit

try:
    import pennylane as qml
except Exception as exc:
    raise RuntimeError("PennyLane is required: !pip -q install pennylane") from exc

try:
    import medmnist
    from medmnist import INFO
except Exception:
    medmnist, INFO = None, None

warnings.filterwarnings("ignore")

SEED = 42

def set_global_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_global_seed(SEED)
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception:
    pass

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_OUTPUT = Path("/content/drive/MyDrive/Outputs/FreeEnergyQuantumPneumonia")
else:
    BASE_OUTPUT = Path.cwd() / "Outputs" / "FreeEnergyQuantumPneumonia"

FIG_DIR = BASE_OUTPUT / "figures"
TAB_DIR = BASE_OUTPUT / "tables"
MODEL_DIR = BASE_OUTPUT / "models"
OTH_DIR = BASE_OUTPUT / "others"
for directory in (BASE_OUTPUT, FIG_DIR, TAB_DIR, MODEL_DIR, OTH_DIR):
    directory.mkdir(parents=True, exist_ok=True)

OUTPUTS_SUMMARY = BASE_OUTPUT / "outputs_summary.txt"

def log_header(title):
    message = f"\n{'=' * 100}\n{title}\n{'=' * 100}\n"
    print(message)
    with OUTPUTS_SUMMARY.open("a", encoding="utf-8") as stream:
        stream.write(message)

def log_text(text=""):
    print(text)
    with OUTPUTS_SUMMARY.open("a", encoding="utf-8") as stream:
        stream.write(str(text) + "\n")

def log_json(obj):
    text = json.dumps(obj, indent=2, default=str)
    print(text)
    with OUTPUTS_SUMMARY.open("a", encoding="utf-8") as stream:
        stream.write(text + "\n")

OUTPUTS_SUMMARY.write_text("CORRECTED STEP-2 NOTEBOOK LOG\n", encoding="utf-8")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

log_header("INITIALIZATION COMPLETE")
log_text(f"Device: {DEVICE}")
log_text(f"Output directory: {BASE_OUTPUT}")
log_text(f"Python: {platform.python_version()}")
log_text(f"PyTorch: {torch.__version__}")
log_text(f"PennyLane: {qml.__version__}")
log_text(f"MedMNIST: {medmnist.__version__ if medmnist is not None else 'not installed; local NPZ mode available'}")


## 2. Configuration

The principal score is

\[
\mathcal{F}(x)= -\frac{1}{T}\sum_{t=1}^{T}\log c_t + \lambda H(\bar p),
\]

where \(c_t\) is the probability assigned during stochastic pass \(t\) to the model's consensus class. No diagnostic label is used in this score. The `target_coverage` value is used to select thresholds on validation data only.


In [ ]:
CONFIG = {
    "data_flag": "pneumoniamnist",
    "download": True,
    "batch_size": 64,
    "num_workers": 0,
    "max_train_samples": 3000,
    "max_val_samples": 600,
    "max_test_samples": 600,
    "latent_dim": 8,
    "dropout_p": 0.25,
    "epochs": 15,
    "min_epochs": 8,
    "early_stopping_patience": 4,
    "lr": 1e-3,
    "weight_decay": 1e-5,
    "mc_passes": 20,
    "free_energy_lambda": 0.2,
    "target_coverage": 0.75,
    "quantum_qubits": 4,
    "quantum_layers": 1,
    "run_classical_control": True,
    "ece_bins": 10,
}

assert 0 < CONFIG["target_coverage"] <= 1
assert CONFIG["mc_passes"] >= 2
assert CONFIG["latent_dim"] >= CONFIG["quantum_qubits"]

log_header("CONFIGURATION")
log_json(CONFIG)


## 3. Dataset loading and deterministic stratified subsetting

PneumoniaMNIST's official train, validation, and test partitions are retained. When a maximum sample count is requested, a deterministic stratified subset is drawn instead of taking the first records. The validation set is reserved for model selection and triage-threshold selection; test labels are used only after the acceptance decision has been produced.


In [ ]:
class NpzImageDataset(Dataset):
    def __init__(self, images, labels):
        self.images = np.asarray(images)
        self.labels = np.asarray(labels).reshape(-1, 1)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        image = torch.from_numpy(self.images[index]).float().unsqueeze(0) / 255.0
        label = torch.tensor(self.labels[index], dtype=torch.long)
        return image, label

npz_candidates = [
    Path("pneumoniamnist.npz"),
    Path("upload/pneumoniamnist.npz"),
    Path("/content/pneumoniamnist.npz"),
    Path("/content/drive/MyDrive/Datasets/pneumoniamnist.npz"),
]
LOCAL_NPZ = next((path for path in npz_candidates if path.exists()), None)

if LOCAL_NPZ is not None:
    archive = np.load(LOCAL_NPZ)
    train_full = NpzImageDataset(archive["train_images"], archive["train_labels"])
    val_full = NpzImageDataset(archive["val_images"], archive["val_labels"])
    test_full = NpzImageDataset(archive["test_images"], archive["test_labels"])
    log_text(f"Dataset source: {LOCAL_NPZ.resolve()}")
else:
    if medmnist is None:
        raise RuntimeError("Attach pneumoniamnist.npz or install medmnist and torchvision.")
    from torchvision import transforms
    info = INFO[CONFIG["data_flag"]]
    DataClass = getattr(medmnist, info["python_class"])
    transform = transforms.Compose([transforms.ToTensor()])
    train_full = DataClass(split="train", transform=transform, download=CONFIG["download"])
    val_full = DataClass(split="val", transform=transform, download=CONFIG["download"])
    test_full = DataClass(split="test", transform=transform, download=CONFIG["download"])

def dataset_labels(dataset):
    if hasattr(dataset, "labels"):
        return np.asarray(dataset.labels).reshape(-1).astype(int)
    labels = []
    for _, y in dataset:
        labels.append(int(np.asarray(y).reshape(-1)[0]))
    return np.asarray(labels, dtype=int)

def stratified_subset(dataset, max_samples, seed):
    if max_samples is None or max_samples >= len(dataset):
        return dataset
    labels = dataset_labels(dataset)
    splitter = StratifiedShuffleSplit(n_splits=1, train_size=max_samples, random_state=seed)
    indices, _ = next(splitter.split(np.zeros(len(labels)), labels))
    return Subset(dataset, indices.tolist())

train_dataset = stratified_subset(train_full, CONFIG["max_train_samples"], SEED)
val_dataset = stratified_subset(val_full, CONFIG["max_val_samples"], SEED + 1)
test_dataset = stratified_subset(test_full, CONFIG["max_test_samples"], SEED + 2)

def make_loader(dataset, shuffle=False, seed=SEED):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        dataset,
        batch_size=CONFIG["batch_size"],
        shuffle=shuffle,
        num_workers=CONFIG["num_workers"],
        generator=generator,
        pin_memory=torch.cuda.is_available(),
    )

val_loader = make_loader(val_dataset, shuffle=False, seed=SEED + 10)
test_loader = make_loader(test_dataset, shuffle=False, seed=SEED + 11)

def class_counts(dataset):
    labels = dataset_labels(dataset.dataset)[dataset.indices] if isinstance(dataset, Subset) else dataset_labels(dataset)
    values, counts = np.unique(labels, return_counts=True)
    return {int(v): int(c) for v, c in zip(values, counts)}

log_header("DATASET")
for name, dataset in (("train", train_dataset), ("validation", val_dataset), ("test", test_dataset)):
    log_text(f"{name}: n={len(dataset)}, class_counts={class_counts(dataset)}")


## 4. Visual inspection

Representative training images are displayed and saved. This plot is descriptive only and is not used to support clinical claims.


In [ ]:
log_header("SAMPLE IMAGES")
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for index, axis in enumerate(axes.flatten()):
    image, label = train_dataset[index]
    label_value = int(np.asarray(label).reshape(-1)[0])
    axis.imshow(image.squeeze().numpy(), cmap="gray")
    axis.set_title(f"Label: {label_value}")
    axis.axis("off")
plt.tight_layout()
sample_path = FIG_DIR / "step2_sample_images.png"
plt.savefig(sample_path, dpi=300, bbox_inches="tight")
plt.show()
log_text(f"Saved: {sample_path}")


## 5. Model definitions and classical control

All models use the same encoder architecture. The hybrid model augments four latent variables with four simulated quantum expectation values. The classical control uses a trainable four-dimensional nonlinear map and the same fusion-classifier dimensions as the hybrid model. Encoder weights are initialized identically across models to reduce initialization-related confounding.


In [ ]:
class Encoder(nn.Module):
    def __init__(self, latent_dim=8, dropout_p=0.25):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.fc = nn.Sequential(
            nn.Flatten(), nn.Linear(64, 32), nn.ReLU(),
            nn.Dropout(dropout_p), nn.Linear(32, latent_dim),
        )

    def forward(self, x):
        return self.fc(self.features(x))

class BaselineNet(nn.Module):
    def __init__(self, latent_dim=8, dropout_p=0.25):
        super().__init__()
        self.encoder = Encoder(latent_dim, dropout_p)
        self.dropout = nn.Dropout(dropout_p)
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 16), nn.ReLU(), nn.Dropout(dropout_p), nn.Linear(16, 2)
        )

    def forward(self, x, return_latent=False):
        z = self.encoder(x)
        logits = self.classifier(self.dropout(z))
        return (logits, z) if return_latent else logits

class ClassicalControlNet(nn.Module):
    def __init__(self, latent_dim=8, feature_dim=4, dropout_p=0.25):
        super().__init__()
        self.encoder = Encoder(latent_dim, dropout_p)
        self.feature_dim = feature_dim
        self.feature_map = nn.Sequential(nn.Linear(feature_dim, feature_dim, bias=False), nn.Tanh())
        self.dropout = nn.Dropout(dropout_p)
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim + feature_dim, 16), nn.ReLU(),
            nn.Dropout(dropout_p), nn.Linear(16, 2)
        )

    def forward(self, x, return_latent=False):
        z = self.encoder(x)
        extra = self.feature_map(z[:, :self.feature_dim])
        z_aug = torch.cat([z, extra], dim=1)
        logits = self.classifier(self.dropout(z_aug))
        return (logits, z_aug) if return_latent else logits

class QuantumLayer(nn.Module):
    def __init__(self, n_qubits=4, n_layers=1):
        super().__init__()
        self.n_qubits = n_qubits
        device = qml.device("default.qubit", wires=n_qubits)

        @qml.qnode(device, interface="torch")
        def circuit(inputs, weights):
            qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation="Y")
            qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
            return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

        self.qlayer = qml.qnn.TorchLayer(
            circuit, {"weights": (n_layers, n_qubits, 3)}
        )

    def forward(self, x):
        if x.ndim != 2 or x.shape[1] != self.n_qubits:
            raise ValueError(f"Expected [batch,{self.n_qubits}], received {tuple(x.shape)}")
        return torch.stack([self.qlayer(row) for row in x], dim=0)

class HybridQuantumNet(nn.Module):
    def __init__(self, latent_dim=8, quantum_qubits=4, quantum_layers=1, dropout_p=0.25):
        super().__init__()
        self.encoder = Encoder(latent_dim, dropout_p)
        self.quantum_qubits = quantum_qubits
        self.quantum = QuantumLayer(quantum_qubits, quantum_layers)
        self.dropout = nn.Dropout(dropout_p)
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim + quantum_qubits, 16), nn.ReLU(),
            nn.Dropout(dropout_p), nn.Linear(16, 2)
        )

    def forward(self, x, return_latent=False):
        z = self.encoder(x)
        encoded = torch.tanh(z[:, :self.quantum_qubits]) * (math.pi / 2)
        quantum_features = self.quantum(encoded)
        z_aug = torch.cat([z, quantum_features], dim=1)
        logits = self.classifier(self.dropout(z_aug))
        return (logits, z_aug) if return_latent else logits

set_global_seed(SEED)
reference_encoder = Encoder(CONFIG["latent_dim"], CONFIG["dropout_p"])
REFERENCE_ENCODER_STATE = copy.deepcopy(reference_encoder.state_dict())

def build_model(model_name):
    set_global_seed(SEED)
    if model_name == "baseline":
        model = BaselineNet(CONFIG["latent_dim"], CONFIG["dropout_p"])
    elif model_name == "classical_control":
        model = ClassicalControlNet(CONFIG["latent_dim"], CONFIG["quantum_qubits"], CONFIG["dropout_p"])
    elif model_name == "hybrid_quantum":
        model = HybridQuantumNet(
            CONFIG["latent_dim"], CONFIG["quantum_qubits"],
            CONFIG["quantum_layers"], CONFIG["dropout_p"]
        )
    else:
        raise KeyError(model_name)
    model.encoder.load_state_dict(copy.deepcopy(REFERENCE_ENCODER_STATE))
    return model.to(DEVICE)

model_names = ["baseline", "hybrid_quantum"]
if CONFIG["run_classical_control"]:
    model_names.insert(1, "classical_control")

def trainable_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

parameter_counts = {name: trainable_parameters(build_model(name)) for name in model_names}
log_header("MODEL PARAMETER COUNTS")
log_json(parameter_counts)


## 6. Training, calibration, and selective-prediction utilities

The utilities below correct two central methodological problems. First, classification ECE is computed from the confidence assigned to the predicted class, not directly from the positive-class probability. Second, the free-energy score is constructed exclusively from MC-dropout predictions. Validation thresholds are later applied unchanged to the test scores.


In [ ]:
def prepare_targets(y):
    if isinstance(y, list):
        y = torch.tensor(y)
    y = y.squeeze()
    return y.unsqueeze(0).long() if y.ndim == 0 else y.long()

def metric_bundle(y_true, y_pred, y_prob):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "auc": float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else np.nan,
    }

def train_one_epoch(model, loader, optimizer):
    model.train()
    losses, predictions, probabilities, targets = [], [], [], []
    for x, y in loader:
        x, y = x.to(DEVICE), prepare_targets(y).to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        loss.backward()
        optimizer.step()
        prob = torch.softmax(logits, dim=1)[:, 1]
        losses.append(float(loss.item()))
        predictions.extend(torch.argmax(logits, dim=1).detach().cpu().tolist())
        probabilities.extend(prob.detach().cpu().tolist())
        targets.extend(y.detach().cpu().tolist())
    metrics = metric_bundle(np.asarray(targets), np.asarray(predictions), np.asarray(probabilities))
    metrics["loss"] = float(np.mean(losses))
    return metrics

@torch.no_grad()
def evaluate_deterministic(model, loader):
    model.eval()
    losses, predictions, probabilities, targets = [], [], [], []
    for x, y in loader:
        x, y = x.to(DEVICE), prepare_targets(y).to(DEVICE)
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        prob = torch.softmax(logits, dim=1)[:, 1]
        losses.append(float(loss.item()))
        predictions.extend(torch.argmax(logits, dim=1).cpu().tolist())
        probabilities.extend(prob.cpu().tolist())
        targets.extend(y.cpu().tolist())
    y_true = np.asarray(targets)
    y_pred = np.asarray(predictions)
    y_prob = np.asarray(probabilities)
    metrics = metric_bundle(y_true, y_pred, y_prob)
    metrics.update({"loss": float(np.mean(losses)), "targets": y_true, "preds": y_pred, "probs": y_prob})
    return metrics

def enable_dropout(model):
    for module in model.modules():
        if isinstance(module, nn.Dropout):
            module.train()

def classification_ece(y_true, y_prob, n_bins=10):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    y_pred = (y_prob >= 0.5).astype(int)
    confidence = np.where(y_pred == 1, y_prob, 1.0 - y_prob)
    correct = (y_pred == y_true).astype(float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for index in range(n_bins):
        if index == n_bins - 1:
            mask = (confidence >= edges[index]) & (confidence <= edges[index + 1])
        else:
            mask = (confidence >= edges[index]) & (confidence < edges[index + 1])
        if mask.any():
            ece += mask.mean() * abs(correct[mask].mean() - confidence[mask].mean())
    return float(ece)

def wilson_interval(successes, total, z=1.959963984540054):
    if total == 0:
        return (np.nan, np.nan)
    proportion = successes / total
    denominator = 1 + z * z / total
    center = (proportion + z * z / (2 * total)) / denominator
    half = z * math.sqrt(proportion * (1 - proportion) / total + z * z / (4 * total * total)) / denominator
    return (float(max(0, center - half)), float(min(1, center + half)))

def label_free_scores_from_mc(mc_probs, lambda_fe=0.2, classification_threshold=0.5):
    # No target-label argument is accepted by design.
    mean_prob = mc_probs.mean(dim=0)
    variance = mc_probs.var(dim=0, unbiased=True)
    entropy = -(mean_prob * torch.log(mean_prob + 1e-8) + (1 - mean_prob) * torch.log(1 - mean_prob + 1e-8))
    prediction = (mean_prob >= classification_threshold).long()
    consensus_probability = torch.where(prediction.unsqueeze(0) == 1, mc_probs, 1 - mc_probs).clamp_min(1e-8)
    energy = -torch.log(consensus_probability).mean(dim=0)
    free_energy = energy + lambda_fe * entropy
    return mean_prob, variance, entropy, prediction, energy, free_energy

@torch.no_grad()
def mc_dropout_predict(model, loader, mc_passes=20, lambda_fe=0.2):
    model.eval()
    enable_dropout(model)
    collected = {key: [] for key in (
        "targets", "mean_probs", "var_probs", "entropy", "energy",
        "free_energy", "preds", "images", "latent", "mc_probabilities"
    )}

    for x, y in loader:
        x, y = x.to(DEVICE), prepare_targets(y).to(DEVICE)
        pass_probs = []
        for _ in range(mc_passes):
            logits, _ = model(x, return_latent=True)
            pass_probs.append(torch.softmax(logits, dim=1)[:, 1].unsqueeze(0))
        mc_probs = torch.cat(pass_probs, dim=0)  # [T, B]
        mean_prob, variance, entropy, prediction, energy, free_energy = label_free_scores_from_mc(mc_probs, lambda_fe)

        model.eval()
        _, deterministic_latent = model(x, return_latent=True)
        enable_dropout(model)

        collected["targets"].append(y.cpu().numpy())
        collected["mean_probs"].append(mean_prob.cpu().numpy())
        collected["var_probs"].append(variance.cpu().numpy())
        collected["entropy"].append(entropy.cpu().numpy())
        collected["energy"].append(energy.cpu().numpy())
        collected["free_energy"].append(free_energy.cpu().numpy())
        collected["preds"].append(prediction.cpu().numpy())
        collected["images"].append(x.cpu().numpy())
        collected["latent"].append(deterministic_latent.cpu().numpy())
        collected["mc_probabilities"].append(mc_probs.transpose(0, 1).cpu().numpy())

    return {key: np.concatenate(value, axis=0) for key, value in collected.items()}

def validation_threshold(scores, target_coverage):
    try:
        return float(np.quantile(scores, target_coverage, method="higher"))
    except TypeError:  # NumPy < 1.22
        return float(np.quantile(scores, target_coverage, interpolation="higher"))

def validation_classification_threshold(y_true, y_prob):
    # Youden's J maximizes validation sensitivity + specificity - 1.
    false_positive_rate, true_positive_rate, thresholds = roc_curve(y_true, y_prob)
    threshold = float(thresholds[np.argmax(true_positive_rate - false_positive_rate)])
    return threshold if np.isfinite(threshold) else 0.5

def refresh_label_free_fields(mc_result, classification_threshold, lambda_fe):
    mc_tensor = torch.as_tensor(mc_result["mc_probabilities"].T, dtype=torch.float32)
    mean_prob, variance, entropy, prediction, energy, free_energy = label_free_scores_from_mc(
        mc_tensor, lambda_fe, classification_threshold
    )
    mc_result["mean_probs"] = mean_prob.numpy()
    mc_result["var_probs"] = variance.numpy()
    mc_result["entropy"] = entropy.numpy()
    mc_result["preds"] = prediction.numpy()
    mc_result["energy"] = energy.numpy()
    mc_result["free_energy"] = free_energy.numpy()

def selective_metrics(y_true, y_pred, accepted):
    accepted = np.asarray(accepted, dtype=bool)
    count = int(accepted.sum())
    coverage = float(accepted.mean())
    if count == 0:
        return {
            "coverage": coverage, "accepted_n": 0, "referred_n": int(len(accepted)),
            "risk": np.nan, "accepted_acc": np.nan, "accepted_f1": np.nan,
            "accepted_acc_ci_low": np.nan, "accepted_acc_ci_high": np.nan,
        }
    correct = int((y_true[accepted] == y_pred[accepted]).sum())
    accuracy = correct / count
    ci_low, ci_high = wilson_interval(correct, count)
    return {
        "coverage": coverage,
        "accepted_n": count,
        "referred_n": int(len(accepted) - count),
        "risk": float(1 - accuracy),
        "accepted_acc": float(accuracy),
        "accepted_f1": float(f1_score(y_true[accepted], y_pred[accepted], zero_division=0)),
        "accepted_acc_ci_low": ci_low,
        "accepted_acc_ci_high": ci_high,
    }

def exact_rank_mask(scores, coverage):
    count = max(1, min(len(scores), int(round(coverage * len(scores)))))
    order = np.argsort(scores, kind="mergesort")
    mask = np.zeros(len(scores), dtype=bool)
    mask[order[:count]] = True
    return mask

# Lightweight score preflight: no target label is passed or available.
toy_mc = torch.tensor([[0.10, 0.55], [0.20, 0.65], [0.15, 0.60]], dtype=torch.float32)
toy_outputs = label_free_scores_from_mc(toy_mc, CONFIG["free_energy_lambda"], 0.6)
assert all(torch.isfinite(value).all() for value in toy_outputs[:-1])
assert torch.isfinite(toy_outputs[-1]).all()
log_text("Label-free score preflight passed.")


## 7. Model training with best-checkpoint restoration

Each model is trained with the same data order and encoder initialization. The checkpoint with the highest validation AUC is restored before uncertainty evaluation. Validation loss is used as a tie-breaker. This prevents a favorable or unfavorable final epoch from being treated automatically as the model result.


In [ ]:
log_header("TRAINING MODELS")
models, history_frames, best_records = {}, {}, {}

for model_index, model_name in enumerate(model_names):
    log_header(f"TRAINING {model_name.upper()}")
    model = build_model(model_name)
    train_loader = make_loader(train_dataset, shuffle=True, seed=SEED + 100)
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
    rows = []
    best_state, best_auc, best_loss, best_epoch = None, -np.inf, np.inf, None

    epochs_without_improvement = 0
    for epoch in range(1, CONFIG["epochs"] + 1):
        started = time.time()
        train_stats = train_one_epoch(model, train_loader, optimizer)
        val_stats = evaluate_deterministic(model, val_loader)
        elapsed = time.time() - started
        row = {
            "model": model_name, "epoch": epoch,
            "train_loss": train_stats["loss"], "train_acc": train_stats["acc"],
            "train_f1": train_stats["f1"], "train_auc": train_stats["auc"],
            "val_loss": val_stats["loss"], "val_acc": val_stats["acc"],
            "val_f1": val_stats["f1"], "val_auc": val_stats["auc"],
            "seconds": elapsed,
        }
        rows.append(row)
        log_json(row)

        candidate_auc = val_stats["auc"] if np.isfinite(val_stats["auc"]) else -np.inf
        if candidate_auc > best_auc or (np.isclose(candidate_auc, best_auc) and val_stats["loss"] < best_loss):
            best_auc, best_loss, best_epoch = candidate_auc, val_stats["loss"], epoch
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epoch >= CONFIG["min_epochs"] and epochs_without_improvement >= CONFIG["early_stopping_patience"]:
            log_text(f"Early stopping {model_name} at epoch {epoch}; best epoch={best_epoch}.")
            break

    if best_state is None:
        raise RuntimeError(f"No valid checkpoint was produced for {model_name}")
    model.load_state_dict(best_state)
    checkpoint_path = MODEL_DIR / f"best_{model_name}.pt"
    torch.save({"model_state_dict": best_state, "config": CONFIG, "best_epoch": best_epoch}, checkpoint_path)
    models[model_name] = model
    history_frames[model_name] = pd.DataFrame(rows)
    best_records[model_name] = {"best_epoch": best_epoch, "best_val_auc": best_auc, "best_val_loss": best_loss}

log_header("BEST CHECKPOINTS")
log_json(best_records)


## 8. Label-free uncertainty and triage evaluation

MC-dropout predictions are computed separately for validation and test sets. Entropy and free-energy thresholds are learned from validation scores at the requested coverage and then applied unchanged to test cases. A second, explicitly retrospective matched-coverage analysis ranks the test cases only to compare score ordering; it must not be described as a prospective operating policy.


In [ ]:
log_header("LABEL-FREE UNCERTAINTY AND TRIAGE EVALUATION")
results = {}
summary_rows = []
matched_rows = []
curve_rows = []

for model_name, model in models.items():
    log_header(f"MC DROPOUT: {model_name.upper()}")
    validation_mc = mc_dropout_predict(
        model, val_loader, CONFIG["mc_passes"], CONFIG["free_energy_lambda"]
    )
    test_mc = mc_dropout_predict(
        model, test_loader, CONFIG["mc_passes"], CONFIG["free_energy_lambda"]
    )

    classification_threshold = validation_classification_threshold(
        validation_mc["targets"], validation_mc["mean_probs"]
    )
    refresh_label_free_fields(validation_mc, classification_threshold, CONFIG["free_energy_lambda"])
    refresh_label_free_fields(test_mc, classification_threshold, CONFIG["free_energy_lambda"])

    entropy_threshold = validation_threshold(validation_mc["entropy"], CONFIG["target_coverage"])
    free_energy_threshold = validation_threshold(validation_mc["free_energy"], CONFIG["target_coverage"])
    validation_accept_entropy = validation_mc["entropy"] <= entropy_threshold
    validation_accept_fe = validation_mc["free_energy"] <= free_energy_threshold
    test_accept_entropy = test_mc["entropy"] <= entropy_threshold
    test_accept_fe = test_mc["free_energy"] <= free_energy_threshold

    overall = metric_bundle(test_mc["targets"], test_mc["preds"], test_mc["mean_probs"])
    overall.update({
        "balanced_accuracy": float(balanced_accuracy_score(test_mc["targets"], test_mc["preds"])),
        "ece": classification_ece(test_mc["targets"], test_mc["mean_probs"], CONFIG["ece_bins"]),
        "brier": float(brier_score_loss(test_mc["targets"], test_mc["mean_probs"])),
        "mean_entropy": float(test_mc["entropy"].mean()),
        "mean_variance": float(test_mc["var_probs"].mean()),
        "mean_energy": float(test_mc["energy"].mean()),
        "mean_free_energy": float(test_mc["free_energy"].mean()),
    })
    operational_entropy = selective_metrics(test_mc["targets"], test_mc["preds"], test_accept_entropy)
    operational_fe = selective_metrics(test_mc["targets"], test_mc["preds"], test_accept_fe)

    target = CONFIG["target_coverage"]
    matched_entropy_mask = exact_rank_mask(test_mc["entropy"], target)
    matched_fe_mask = exact_rank_mask(test_mc["free_energy"], target)
    matched_entropy = selective_metrics(test_mc["targets"], test_mc["preds"], matched_entropy_mask)
    matched_fe = selective_metrics(test_mc["targets"], test_mc["preds"], matched_fe_mask)

    results[model_name] = {
        "validation": validation_mc, "test": test_mc,
        "entropy_threshold": entropy_threshold,
        "free_energy_threshold": free_energy_threshold,
        "validation_entropy_coverage": float(validation_accept_entropy.mean()),
        "validation_free_energy_coverage": float(validation_accept_fe.mean()),
        "test_accept_entropy": test_accept_entropy,
        "test_accept_free_energy": test_accept_fe,
        "overall": overall,
        "classification_threshold": classification_threshold,
        "operational_entropy": operational_entropy,
        "operational_free_energy": operational_fe,
        "matched_entropy": matched_entropy,
        "matched_free_energy": matched_fe,
    }

    summary_rows.append({
        "model": model_name, **overall,
        "classification_threshold_validation": classification_threshold,
        "entropy_threshold_validation": entropy_threshold,
        "free_energy_threshold_validation": free_energy_threshold,
        **{f"entropy_{key}": value for key, value in operational_entropy.items()},
        **{f"free_energy_{key}": value for key, value in operational_fe.items()},
    })
    for criterion, metrics in (("entropy", matched_entropy), ("free_energy", matched_fe)):
        matched_rows.append({"model": model_name, "criterion": criterion, "analysis": "retrospective_exact_test_coverage", **metrics})

    for coverage in np.linspace(0.10, 1.00, 19):
        for criterion, score in (("entropy", test_mc["entropy"]), ("free_energy", test_mc["free_energy"])):
            metrics = selective_metrics(test_mc["targets"], test_mc["preds"], exact_rank_mask(score, coverage))
            curve_rows.append({"model": model_name, "criterion": criterion, "requested_coverage": coverage, **metrics})

summary_df = pd.DataFrame(summary_rows)
matched_coverage_df = pd.DataFrame(matched_rows)
coverage_risk_df = pd.DataFrame(curve_rows)

log_text("\nPRIMARY TEST SUMMARY: VALIDATION-LOCKED THRESHOLDS")
log_text(summary_df.round(6).to_string(index=False))
log_text("\nRETROSPECTIVE MATCHED-COVERAGE DIAGNOSTIC")
log_text(matched_coverage_df.round(6).to_string(index=False))


## 9. Diagnostic figures

Reliability plots display the observed pneumonia frequency against the mean predicted pneumonia probability. Coverage--risk curves are retrospective ranking diagnostics and are labeled accordingly. The validation-locked operating-point results are retained in the summary table rather than being conflated with these curves.


In [ ]:
log_header("DIAGNOSTIC FIGURES")

plt.figure(figsize=(7, 4.5))
for model_name, frame in history_frames.items():
    plt.plot(frame["epoch"], frame["val_acc"], marker="o", label=model_name)
plt.xlabel("Epoch")
plt.ylabel("Validation accuracy")
plt.title("Validation Accuracy Across Epochs")
plt.legend()
plt.tight_layout()
path = FIG_DIR / "step2_val_accuracy.png"
plt.savefig(path, dpi=300, bbox_inches="tight")
plt.show()

def plot_probability_reliability(y_true, y_prob, title, save_path, n_bins=10):
    edges = np.linspace(0, 1, n_bins + 1)
    mean_probability, fraction_positive = [], []
    for index in range(n_bins):
        upper_rule = y_prob <= edges[index + 1] if index == n_bins - 1 else y_prob < edges[index + 1]
        mask = (y_prob >= edges[index]) & upper_rule
        if mask.any():
            mean_probability.append(y_prob[mask].mean())
            fraction_positive.append(y_true[mask].mean())
    plt.figure(figsize=(5, 5))
    plt.plot([0, 1], [0, 1], "--", color="gray", label="Ideal")
    plt.plot(mean_probability, fraction_positive, marker="o", label="Model")
    plt.xlabel("Mean predicted P(pneumonia)")
    plt.ylabel("Observed pneumonia frequency")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

for model_name, result in results.items():
    plot_probability_reliability(
        result["test"]["targets"], result["test"]["mean_probs"],
        f"Probability Reliability: {model_name}",
        FIG_DIR / f"step2_reliability_{model_name}.png",
        CONFIG["ece_bins"],
    )

plt.figure(figsize=(7, 5))
for (model_name, criterion), frame in coverage_risk_df.groupby(["model", "criterion"]):
    style = "-" if criterion == "free_energy" else "--"
    plt.plot(frame["coverage"], frame["risk"], style, marker="o", markersize=3, label=f"{model_name} | {criterion}")
plt.xlabel("Coverage")
plt.ylabel("Selective risk")
plt.title("Retrospective Coverage--Risk Curves")
plt.legend(fontsize=8)
plt.tight_layout()
path = FIG_DIR / "step2_coverage_risk.png"
plt.savefig(path, dpi=300, bbox_inches="tight")
plt.show()

for model_name, result in results.items():
    test = result["test"]
    plt.figure(figsize=(6, 4))
    plt.scatter(test["entropy"], test["free_energy"], c=(test["preds"] != test["targets"]), cmap="coolwarm", alpha=0.55, s=14)
    plt.xlabel("Predictive entropy")
    plt.ylabel("Label-free free energy")
    plt.title(f"Entropy vs Free Energy: {model_name}")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"step2_entropy_vs_free_energy_{model_name}.png", dpi=300, bbox_inches="tight")
    plt.show()


## 10. Latent-space visualization

A deterministic representation is sampled reproducibly for t-SNE. The projection is retained as a qualitative diagnostic only; it is not interpreted as independent evidence of class separation or quantum advantage.


In [ ]:
log_header("LATENT-SPACE VISUALIZATION")
rng = np.random.default_rng(SEED)

for model_name, result in results.items():
    latent = result["test"]["latent"]
    targets = result["test"]["targets"]
    count = min(400, len(latent))
    indices = np.sort(rng.choice(len(latent), size=count, replace=False))
    perplexity = min(30, max(5, count // 10))
    projection = TSNE(n_components=2, random_state=SEED, perplexity=perplexity, init="pca").fit_transform(latent[indices])
    plt.figure(figsize=(6, 5))
    scatter = plt.scatter(projection[:, 0], projection[:, 1], c=targets[indices], cmap="coolwarm", s=18, alpha=0.8)
    plt.title(f"t-SNE Diagnostic: {model_name}")
    plt.xlabel("t-SNE 1")
    plt.ylabel("t-SNE 2")
    plt.colorbar(scatter, ticks=[0, 1], label="Class")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"step2_tsne_{model_name}.png", dpi=300, bbox_inches="tight")
    plt.show()


## 11. Case-level inspection

High-entropy cases are displayed for error analysis. Ground-truth labels are shown only in this retrospective evaluation figure and never enter the uncertainty score or acceptance decision.


In [ ]:
log_header("CASE-LEVEL INSPECTION")

for model_name, result in results.items():
    test = result["test"]
    indices = np.argsort(-test["entropy"])[:8]
    fig, axes = plt.subplots(2, 4, figsize=(11, 5.5))
    for axis, index in zip(axes.flatten(), indices):
        status = "accept" if result["test_accept_free_energy"][index] else "refer"
        axis.imshow(test["images"][index].squeeze(), cmap="gray")
        axis.set_title(
            f"T={test['targets'][index]} P={test['preds'][index]} | {status}\n"
            f"p={test['mean_probs'][index]:.2f}, H={test['entropy'][index]:.3f}, F={test['free_energy'][index]:.3f}",
            fontsize=8,
        )
        axis.axis("off")
    plt.suptitle(f"Highest-Entropy Test Cases: {model_name}")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"step2_uncertain_cases_{model_name}.png", dpi=300, bbox_inches="tight")
    plt.show()


## 12. Persisting tables, per-case predictions, and audit metadata

All aggregate and per-case values required to update the manuscript are saved. The MC probability matrix is stored as compressed NumPy data, allowing the free-energy equation and alternative \(\lambda\) values to be checked without retraining. The audit metadata records that the score is label-free and that operational thresholds were selected on validation data.


In [ ]:
log_header("SAVING FINAL ARTIFACTS")

history_df = pd.concat(history_frames.values(), ignore_index=True)
history_df.to_csv(TAB_DIR / "step2_history.csv", index=False)
summary_df.to_csv(TAB_DIR / "step2_summary_validation_locked.csv", index=False)
matched_coverage_df.to_csv(TAB_DIR / "step2_matched_coverage_diagnostic.csv", index=False)
coverage_risk_df.to_csv(TAB_DIR / "step2_coverage_risk_curves.csv", index=False)
pd.DataFrame([{"model": name, "trainable_parameters": count} for name, count in parameter_counts.items()]).to_csv(
    TAB_DIR / "step2_parameter_counts.csv", index=False
)

for model_name, result in results.items():
    for split_name in ("validation", "test"):
        values = result[split_name]
        detail = pd.DataFrame({
            "case_index": np.arange(len(values["targets"])),
            "target_evaluation_only": values["targets"],
            "prediction": values["preds"],
            "mean_probability": values["mean_probs"],
            "predictive_variance": values["var_probs"],
            "predictive_entropy": values["entropy"],
            "consensus_self_information": values["energy"],
            "label_free_free_energy": values["free_energy"],
        })
        if split_name == "test":
            detail["accepted_entropy_validation_threshold"] = result["test_accept_entropy"]
            detail["accepted_free_energy_validation_threshold"] = result["test_accept_free_energy"]
        detail.to_csv(TAB_DIR / f"step2_detail_{split_name}_{model_name}.csv", index=False)
        np.savez_compressed(
            OTH_DIR / f"step2_mc_probabilities_{split_name}_{model_name}.npz",
            mc_probabilities=values["mc_probabilities"],
            targets=values["targets"],
        )

audit_metadata = {
    "score_uses_ground_truth_label": False,
    "score_definition": "mean negative log probability of consensus class across MC passes + lambda * entropy of predictive mean",
    "operational_threshold_source": "validation set",
    "classification_threshold_source": "validation-set Youden J statistic",
    "test_labels_used_for": ["retrospective metrics", "figures", "error analysis"],
    "test_labels_used_for_score_or_acceptance": False,
    "retrospective_curve_warning": "Coverage-risk curves use exact test ranking for diagnostic comparison and are not prospective operating points.",
    "quantum_execution": "PennyLane default.qubit simulator",
    "quantum_advantage_claimed": False,
    "random_seed": SEED,
    "config": CONFIG,
    "best_checkpoints": best_records,
    "parameter_counts": parameter_counts,
}
(OTH_DIR / "methodology_audit.json").write_text(json.dumps(audit_metadata, indent=2), encoding="utf-8")

log_text(f"Saved all artifacts under: {BASE_OUTPUT}")
log_json(audit_metadata)


## 13. Interpretation checklist for manuscript updating

After a complete run, the manuscript should be updated only from the newly generated CSV files.

1. Use `step2_summary_validation_locked.csv` for the primary operational results.
2. Use `step2_matched_coverage_diagnostic.csv` only when explicitly describing a retrospective comparison at identical test coverage.
3. Report the Wilson interval accompanying accepted-case accuracy, especially when zero errors are observed.
4. Compare the hybrid model with `classical_control` before attributing any gain to the simulated quantum transformation.
5. Do not reuse the earlier free-energy table, curve, mean free-energy values, or zero-risk statement.
6. Report that the circuit is simulated and that no quantum computational advantage has been established.
